In [1]:
# ==============================================================================
# SCRIPT:         colab_log_enrichment_DEFINITIVE.ipynb
# OPERATION:      Definitive Log Enrichment
# PURPOSE:        Handles complex Session_IDs and includes all required provenance columns.
#                 This is the final, production-ready script for this mission.
# ==============================================================================

# --- Step 1: Install & Import ---
!pip install -q gspread
import pandas as pd
from google.colab import auth, drive
import gspread
from google.auth import default
import os

# --- Step 2: Authentication & Mounting ---
print("Authenticating user for Google services...")
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)
print("✅ G-Sheets access authorized.")

print("\nMounting Google Drive...")
drive.mount('/content/drive', force_remount=True)
print("✅ Google Drive mounted successfully.")

# --- Step 3: CONFIGURATION (User Provided) ---
# NOTE: Standardized '/MyDrive/' to '/My Drive/' for robustness.
ASSET_LEDGER_PATH = '/content/drive/My Drive/Opus_1_101001/Database/02_Cleaned_Data/chrono_forge_v6_output.csv'
SPREADSHEET_NAME = 'RAW_Requests'
WORKSHEET_NAME = 'raw_requests_VVSpolished'
OUTPUT_FOLDER_PATH = '/content/drive/My Drive/Opus_1_101001/Database/03_analysis/'
OUTPUT_FILENAME = 'raw_requests_enriched_DEFINITIVE.csv' # Definitive output name

# --- UPGRADED MAPPING FUNCTION (Handles hyphenated Session_IDs) ---
def session_id_to_folder(session_id):
    if not isinstance(session_id, str) or not session_id.startswith('SID'):
        return None

    # Take the part before any hyphen to handle SID0005-1 and SID0005-2 correctly.
    base_id = session_id.split('-')[0]

    # Extract only the digits from the base ID.
    number_part_str = ''.join(filter(str.isdigit, base_id))

    if number_part_str:
        # Convert to an integer to remove leading zeros, then format the name.
        number_part_int = int(number_part_str)
        return f"Screenshots_{number_part_int}"

    return session_id # Fallback if no digits found

try:
    # --- Phase 1: Load and Prepare Data Sources ---
    print("\nLoading data sources...")
    df_assets = pd.read_csv(ASSET_LEDGER_PATH)
    worksheet = gc.open(SPREADSHEET_NAME).worksheet(WORKSHEET_NAME)
    df_log = pd.DataFrame(worksheet.get_all_records())

    # Replace empty strings in join key with NaN to prevent bad matches and drop them
    df_log['Image'].replace('', pd.NA, inplace=True)
    df_log.dropna(subset=['Image'], inplace=True)

    # Prepare the log DataFrame with the CORRECTED and ROBUST folder key
    df_log['log_folder_key'] = df_log['Session_ID'].apply(session_id_to_folder)
    print("  > Prepared log file with corrected folder key for joining.")

    # --- Phase 2: Perform the COMPOUND Left Join ---
    print("\nPerforming precise compound join...")
    df_enriched = pd.merge(
        left=df_log,
        right=df_assets, # Join against the full assets to bring in all columns
        left_on=['Image', 'log_folder_key'],
        right_on=['filename', 'containing_folder'],
        how='left'
    )

    # --- MODIFIED: Drop only the temporary helper key, keeping all other columns ---
    df_enriched = df_enriched.drop(columns=['log_folder_key'], errors='ignore')
    print(f"  > Join complete. Final dataset has {len(df_enriched)} rows.")

    # --- Phase 3: Save the Enriched Log ---
    print("\nSaving enriched log file...")
    full_output_path = os.path.join(OUTPUT_FOLDER_PATH, OUTPUT_FILENAME)
    df_enriched.to_csv(full_output_path, index=False)

    print(f"\n✅ Mission Success. Definitive enriched log saved to:\n   > '{full_output_path}'")

    # --- Verification checks ---
    print("\n--- Final Verification ---")
    filled_rows = df_enriched['timestamp'].notna().sum()
    print(f"  > Found {filled_rows} rows with enriched timestamp data.")

    # Display the final required columns for confirmation
    final_check_cols = ['Session_ID', 'Image'] + ['event_id', 'image_content_hash', 'containing_folder', 'filename', 'timestamp']
    successful_joins = df_enriched[df_enriched['event_id'].notna()]
    print("\n--- Sample of Final Enriched Columns (successful joins only) ---")
    if not successful_joins.empty:
        print(successful_joins[final_check_cols].tail().to_string())
    else:
        print("  > No successful joins were found. Please verify data and paths.")


except Exception as e:
    print(f"❌ An unrecoverable error occurred: {e}")

Authenticating user for Google services...
✅ G-Sheets access authorized.

Mounting Google Drive...
Mounted at /content/drive
✅ Google Drive mounted successfully.

Loading data sources...


/tmp/ipython-input-3828262050.py:61: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_log['Image'].replace('', pd.NA, inplace=True)


  > Prepared log file with corrected folder key for joining.

Performing precise compound join...
  > Join complete. Final dataset has 4776 rows.

Saving enriched log file...

✅ Mission Success. Definitive enriched log saved to:
   > '/content/drive/My Drive/Opus_1_101001/Database/03_analysis/raw_requests_enriched_DEFINITIVE.csv'

--- Final Verification ---
  > Found 4766 rows with enriched timestamp data.

--- Sample of Final Enriched Columns (successful joins only) ---
     Session_ID         Image                                                          event_id                                                image_content_hash containing_folder      filename            timestamp
4771    SID0062  IMG_5997.PNG  fa170244daa5456c6662dfe39fcaf9d3cd7673268bf81f583490ee8a5e6ff47d  3ce5f6e3398d545e3f2cb259f08467735396892d6a1b4648082081ff40d4f9ae    Screenshots_62  IMG_5997.PNG  2025:10:01 09:06:38
4772    SID0062  IMG_5999.PNG  b732d111033d78534c5db21fb4f7436c520accf3600c84444cf2882dd40a21b